# Handwritten Formula to LaTeX: Experiments

In [1]:
%pip install -q transformers datasets peft trl bitsandbytes accelerate pillow jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.8 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from PIL import Image
import jiwer
from tqdm import tqdm

In [3]:
try:
    import torch_xla.core.xla_model as xm
    DEVICE = xm.xla_device()
    IS_TPU = True
except ImportError:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    IS_TPU = False

### Constants

In [4]:
PROMPT = (
    "Convert this handwritten math formula to LaTeX."
    "Return ONLY the LaTeX code, nothing else."
)

In [5]:
MODEL = "HuggingFaceTB/SmolVLM-256M-Instruct"
DATASET = "linxy/LaTeX_OCR"
DATASET_MATHWRITING = "deepcopy/MathWriting-human"

In [9]:
NUM_TRAIN_SAMPLES = 1000
NUM_TRAIN_SAMPLES_2 = 400
NUM_EVAL_SAMPLES  = 70
MAX_MATHWRITING = 400

In [7]:
OUTPUT_DIR_V1 = "./smolvlm-latex-lora-v1"
OUTPUT_DIR_V2 = "./smolvlm-latex-lora-v2"

In [30]:
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
MAX_SEQ_LEN = 2048

In [11]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

### Shared functions

In [31]:
def zero_shot_predict(model, processor, image: Image.Image):
    """
    Method that generates answer with no single example
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": PROMPT}
            ]
        }
    ]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=text,
        images=[image],
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )
    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][prompt_len:]
    result = processor.decode(new_tokens, skip_special_tokens=True)
    return result.strip()

In [32]:
def one_shot_predict(
    model, processor,
    image: Image.Image,
    example_image: Image.Image,
    example_formula,
):
    """
    Method that answers using single example
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": PROMPT}
            ]
        },
        {
            "role": "assistant",
            "content": example_formula
        },
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": PROMPT}
            ]
        }
    ]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=text,
        images=[example_image, image],
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN * 2,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )
    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][prompt_len:]
    result = processor.decode(new_tokens, skip_special_tokens=True)
    return result.strip()

In [33]:
def compute_metrics(predictions, references):
    """
    Method that calculates metric to evaluate generation
    """
    total_cer = 0.0
    n = 0

    for pred, ref in zip(predictions, references):
        if len(ref) == 0:
            continue
        total_cer += jiwer.cer(ref, pred)
        n += 1

    return {
        "cer": round(total_cer / n, 4) if n > 0 else 0.0,
        "n_samples": n,
    }

In [34]:
def evaluate_model(
    model, processor,
    test_dataset,
    mode,
    one_shot_example=None,
):
    """
    Method that fully evaluates model
    """
    print(f"\nModel evaluation: {mode}")

    predictions = []
    references = []
    model.eval()

    for i, example in enumerate(tqdm(test_dataset, desc=str({mode}))):
        image = example["image"].convert("RGB")
        reference = example["text"]

        try:
            if mode == "one_shot" and one_shot_example is not None:
                pred = one_shot_predict(
                    model, processor, image,
                    one_shot_example["image"].convert("RGB"),
                    one_shot_example["text"],
                )
            else:
                pred = zero_shot_predict(model, processor, image)

        except Exception as e:
            print(f"\nError on iteration №{i}: {e}")
            pred = ""

        predictions.append(pred)
        references.append(reference)

    metrics = compute_metrics(predictions, references)
    print(f"Result of evaluating {mode}: CER={metrics['cer']}")

    return {
        "metrics": metrics,
        "predictions": predictions,
        "references": references,
    }

In [35]:
def find_linear_names(model):
    """
    Method that finds linear layers for tuning
    """
    names = []
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            names.append(name.split(".")[-1])
    return list(set(names))


def build_lora_model():
    """
    Load model with 4-bit quantization and wrap with LoRA
    """
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL,
        dtype=torch.bfloat16,
        _attn_implementation="eager"
    )
    model.config.use_cache = False

    target_modules = find_linear_names(model)
    target_modules = [m for m in target_modules if m != "lm_head"]

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model


def make_collate_fn(processor):
    """
    Returns a collate_fn bound to the given processor
    """
    def collate_fn(examples):
        """
        Collator
        """
        images = []
        texts = []
        for ex in examples:
            img = ex["image"]
            formula = ex["text"]
            if not isinstance(img, Image.Image):
                img = Image.fromarray(img)
            img = img.convert("RGB")
            images.append(img)
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": PROMPT},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": formula}],
                },
            ]
            text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            texts.append(text)

        batch = processor(
            text=texts,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        )
        labels = batch["input_ids"].clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        image_token_id = processor.tokenizer.convert_tokens_to_ids(processor.image_token)
        labels[labels == image_token_id] = -100
        batch["labels"] = labels
        return batch
    return collate_fn


def build_training_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=0.05,
        fp16=False,
        bf16=True,
        gradient_checkpointing=True,
        logging_steps=20,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        remove_unused_columns=False,
        dataloader_pin_memory=False,
        report_to="none",
    )

### Report generators

In [17]:
def generate_report_zero_shot(r: dict):
    report_lines = [
        "# Tech report: Handwritten Formula to LaTeX\n",
        "\n",
        "## Model\n",
        "\n",
        "- **Name**: SmolVLM-256M-Instruct\n",
        "- **NF Name**: `HuggingFaceTB/SmolVLM-256M-Instruct`\n",
        "- **Approach**: Zero-Shot\n",
        "\n",
        "## Evaluation scores\n",
        "\n",
        "**CER (Character Error Rate) = edit_distance(prediction, reference) / len(reference)**\n",
        "\n",
        "- The closer to zero the better\n",
        "- I decided to take this score as:\n",
        "  - LaTeX is sensible to each symbol\n",
        "  - Easy to interpret\n",
        "\n",
        "## Configuration\n",
        "\n",
        "| Param | Value |\n",
        "|----------|----------|\n",
        f"| Max sequence length | {MAX_SEQ_LEN} |\n",
        "\n",
        "## Evaluation results\n",
        "\n",
        "| Approach | CER |\n",
        "|-------|-------|\n",
        f"| 1. Zero-Shot | {r['metrics']['cer']} |\n",
        "\n",
        "## Links\n",
        "\n",
        "- Model: https://huggingface.co/HuggingFaceTB/SmolVLM-256M-Instruct\n",
        "- Dataset LaTeX_OCR: https://huggingface.co/datasets/linxy/LaTeX_OCR\n",
    ]
    with open("report_1_zero_shot.md", "w", encoding="utf-8") as f:
        f.writelines(report_lines)

In [18]:
def generate_report_one_shot(r: dict):
    report_lines = [
        "# Tech report: Handwritten Formula to LaTeX\n",
        "\n",
        "## Model\n",
        "\n",
        "- **Name**: SmolVLM-256M-Instruct\n",
        "- **HF Name**: `HuggingFaceTB/SmolVLM-256M-Instruct`\n",
        "- **Approach**: One-Shot\n",
        "\n",
        "## Evaluation scores\n",
        "\n",
        "**CER (Character Error Rate) = edit_distance(prediction, reference) / len(reference)**\n",
        "\n",
        "- The closer to zero the better\n",
        "- I decided to take this score as:\n",
        "  - LaTeX is sensible to each symbol\n",
        "  - Easy to interpret\n",
        "\n",
        "## Configuration\n",
        "\n",
        "| Param | Value |\n",
        "|----------|----------|\n",
        f"| Max sequence length | {MAX_SEQ_LEN} |\n",
        "\n",
        "## Evaluation results\n",
        "\n",
        "| Approach | CER |\n",
        "|-------|-------|\n",
        f"| 2. One-Shot | {r['metrics']['cer']} |\n",
        "\n",
        "## Links\n",
        "\n",
        "- Model: https://huggingface.co/HuggingFaceTB/SmolVLM-256M-Instruct\n",
        "- Dataset LaTeX_OCR: https://huggingface.co/datasets/linxy/LaTeX_OCR\n",
    ]
    with open("report_2_one_shot.md", "w", encoding="utf-8") as f:
        f.writelines(report_lines)

In [36]:
def generate_report_sft_v1(r: dict):
    report_lines = [
        "# Tech report: Handwritten Formula to LaTeX\n",
        "\n",
        "## Model\n",
        "\n",
        "- **Name**: SmolVLM-256M-Instruct\n",
        "- **HF Name**: `HuggingFaceTB/SmolVLM-256M-Instruct`\n",
        "- **Fine-tuning approach on single dataset**: LoRA\n",
        "\n",
        "## Evaluation scores\n",
        "\n",
        "**CER (Character Error Rate) = edit_distance(prediction, reference) / len(reference)**\n",
        "\n",
        "- The closer to zero the better\n",
        "- I decided to take this score as:\n",
        "  - LaTeX is sensible to each symbol\n",
        "  - Easy to interpret\n",
        "\n",
        "## Hyperparameters and datasets info\n",
        "\n",
        "### Tuning hyperparameters\n",
        "\n",
        "| Param | Value |\n",
        "|----------|----------|\n",
        f"| Learning rate | {LEARNING_RATE} |\n",
        f"| Effective batch size | {BATCH_SIZE * GRAD_ACCUM} |\n",
        f"| Num epochs | {NUM_EPOCHS} |\n",
        f"| Max sequence length | {MAX_SEQ_LEN} |\n",
        "| LoRA rank | 16 |\n",
        "| LoRA alpha | 32 |\n",
        "| LR scheduler | Cosine |\n",
        "| Warmup steps | 0.05 |\n",
        "| Gradient checkpointing | True |\n",
        "\n",
        "### Datasets\n",
        "\n",
        "| Dataset | Split | Usage |\n",
        "|---------|-------|---------------|\n",
        f"| `linxy/LaTeX_OCR` | train ({NUM_TRAIN_SAMPLES} examples) | Tuning and testing |\n",
        "\n",
        "## Evaluation results\n",
        "\n",
        "| Approach | CER |\n",
        "|-------|-------|\n",
        f"| 3. SFT (LaTeX_OCR only) | {r['metrics']['cer']} |\n",
        "\n",
        "## Links\n",
        "\n",
        "- Model: https://huggingface.co/HuggingFaceTB/SmolVLM-256M-Instruct\n",
        "- Dataset: https://huggingface.co/datasets/linxy/LaTeX_OCR\n",
    ]
    with open("report_3_sft_v1.md", "w", encoding="utf-8") as f:
        f.writelines(report_lines)

In [37]:
def generate_report_sft_v2(r: dict):
    report_lines = [
        "# Tech report: Handwritten Formula to LaTeX\n",
        "\n",
        "## Model\n",
        "\n",
        "- **Name**: SmolVLM-256M-Instruct\n",
        "- **HF Name**: `HuggingFaceTB/SmolVLM-256M-Instruct`\n",
        "- **Fine-tuning approach on 2 concatenated datasets**: LoRA\n",
        "\n",
        "## Evaluation scores\n",
        "\n",
        "**CER (Character Error Rate) = edit_distance(prediction, reference) / len(reference)**\n",
        "\n",
        "- The closer to zero the better\n",
        "- I decided to take this score as:\n",
        "  - LaTeX is sensible to each symbol\n",
        "  - Easy to interpret\n",
        "\n",
        "## Hyperparameters and datasets info\n",
        "\n",
        "### Tuning hyperparameters\n",
        "\n",
        "| Param | Value |\n",
        "|----------|----------|\n",
        f"| Learning rate | {LEARNING_RATE} |\n",
        f"| Effective batch size | {BATCH_SIZE * GRAD_ACCUM} |\n",
        f"| Num epochs | {NUM_EPOCHS} |\n",
        f"| Max sequence length | {MAX_SEQ_LEN} |\n",
        "| LoRA rank | 16 |\n",
        "| LoRA alpha | 32 |\n",
        "| LR scheduler | Cosine |\n",
        "| Warmup steps | 0.05 |\n",
        "| Gradient checkpointing | True |\n",
        "\n",
        "### Datasets\n",
        "\n",
        "| Dataset | Split | Usage |\n",
        "|---------|-------|---------------|\n",
        f"| `linxy/LaTeX_OCR` | train ({NUM_TRAIN_SAMPLES_2} examples) | Tuning and testing |\n",
        f"| `deepcopy/MathWriting-human` | train ({MAX_MATHWRITING} examples) | Tuning |\n",
        "\n",
        "## Evaluation results\n",
        "\n",
        "| Approach | CER |\n",
        "|-------|-------|\n",
        f"| 4. SFT (LaTeX_OCR + MathWriting) | {r['metrics']['cer']} |\n",
        "\n",
        "## Links\n",
        "\n",
        "- Model: https://huggingface.co/HuggingFaceTB/SmolVLM-256M-Instruct\n",
        "- Dataset LaTeX_OCR: https://huggingface.co/datasets/linxy/LaTeX_OCR\n",
        "- Dataset MathWriting: https://huggingface.co/datasets/deepcopy/MathWriting-human\n",
    ]
    with open("report_4_sft_v2.md", "w", encoding="utf-8") as f:
        f.writelines(report_lines)

## Approach 1 — Zero-Shot

In [ ]:
processor_zs = AutoProcessor.from_pretrained(MODEL)
model_zs = AutoModelForImageTextToText.from_pretrained(
    MODEL,
    dtype=torch.bfloat16,
    _attn_implementation="eager",
).to(DEVICE)
model_zs.eval()
print("Zero-shot model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Zero-shot model loaded.


In [ ]:
test_ds_zs = load_dataset(DATASET, split="train").shuffle(seed=42).select(range(NUM_EVAL_SAMPLES))
print(f"Test set size: {len(test_ds_zs)}")

README.md: 0.00B [00:00, ?B/s]

full/train-00000-of-00001.parquet:   0%|          | 0.00/383M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/76318 [00:00<?, ? examples/s]

Test set size: 70


In [ ]:
results_zs = evaluate_model(
    model_zs, processor_zs,
    test_ds_zs,
    mode="zero_shot",
)


Model evaluation: zero_shot


{'zero_shot'}: 100%|██████████| 70/70 [05:27<00:00,  4.68s/it]

Result of evaluating zero_shot: CER=1.3129


In [ ]:
generate_report_zero_shot(results_zs)

## Approach 2 — One-Shot

In [ ]:
processor_os = AutoProcessor.from_pretrained(MODEL)
model_os = AutoModelForImageTextToText.from_pretrained(
    MODEL,
    dtype=torch.bfloat16,
    _attn_implementation="eager",
).to(DEVICE)
model_os.eval()
print("One-shot model loaded.")

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

One-shot model loaded.


In [ ]:
raw_os = load_dataset(DATASET, split="train").shuffle(seed=42)
one_shot_example = raw_os[0]

test_ds_os = raw_os.select(range(1, 1 + NUM_EVAL_SAMPLES))
print(f"Test set size: {len(test_ds_os)}")
print("One-shot example formula:", one_shot_example["text"])

Test set size: 70
One-shot example formula: H _ { \pm } = \frac { 2 c } { \mid \kappa \mid } \left[ \sqrt { 1 + \omega \frac { D - 2 } { D - 1 } } \mp \{ 1 + \omega ( 1 - \gamma ) \} \right] .


In [ ]:
results_os = evaluate_model(
    model_os, processor_os,
    test_ds_os,
    mode="one_shot",
    one_shot_example=one_shot_example,
)


Model evaluation: one_shot


{'one_shot'}: 100%|██████████| 70/70 [07:01<00:00,  6.03s/it]

Result of evaluating one_shot: CER=1.3385


In [ ]:
generate_report_one_shot(results_os)

## Approach 3 — SFT v1 (fine-tuned on LaTeX\_OCR only)

In [ ]:
processor_v1 = AutoProcessor.from_pretrained(MODEL)

In [ ]:
model_v1 = build_lora_model()

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

trainable params: 7,744,512 || all params: 264,229,440 || trainable%: 2.9310


In [ ]:
raw_v1 = load_dataset(DATASET, split="train").shuffle(seed=42)
train_ds_v1 = raw_v1.select(range(NUM_TRAIN_SAMPLES))
eval_ds_v1  = raw_v1.select(range(NUM_TRAIN_SAMPLES, NUM_TRAIN_SAMPLES + NUM_EVAL_SAMPLES))
print(f"Train: {len(train_ds_v1)}, Eval: {len(eval_ds_v1)}")

Train: 1000, Eval: 70


In [ ]:
collate_fn_v1 = make_collate_fn(processor_v1)
test_batch_v1 = collate_fn_v1([train_ds_v1[0], train_ds_v1[1]])
print({k: v.shape for k, v in test_batch_v1.items()})

{'pixel_values': torch.Size([2, 5, 3, 512, 512]), 'pixel_attention_mask': torch.Size([2, 5, 512, 512]), 'input_ids': torch.Size([2, 430]), 'attention_mask': torch.Size([2, 430]), 'labels': torch.Size([2, 430])}


In [ ]:
training_args_v1 = build_training_args(OUTPUT_DIR_V1)

trainer_v1 = SFTTrainer(
    model=model_v1,
    args=training_args_v1,
    train_dataset=train_ds_v1,
    eval_dataset=eval_ds_v1,
    data_collator=collate_fn_v1,
    peft_config=None,
)
trainer_v1.train()

Epoch,Training Loss,Validation Loss
1,0.366007,0.330664


TrainOutput(global_step=125, training_loss=0.6554209051132203, metrics={'train_runtime': 2937.6265, 'train_samples_per_second': 0.34, 'train_steps_per_second': 0.043, 'total_flos': 709556540511744.0, 'train_loss': 0.6554209051132203})

In [ ]:
test_ds_v1 = raw_v1.select(range(
    NUM_TRAIN_SAMPLES + NUM_EVAL_SAMPLES,
    NUM_TRAIN_SAMPLES + NUM_EVAL_SAMPLES + NUM_EVAL_SAMPLES,
))
print(f"Test set size: {len(test_ds_v1)}")

Test set size: 70


In [ ]:
results_v1 = evaluate_model(
    model_v1, processor_v1,
    test_ds_v1,
    mode="zero_shot",
)

In [ ]:
generate_report_sft_v1(results_v1)

## Approach 4 — SFT v2 (fine-tuned on LaTeX\_OCR + MathWriting-human)

In [38]:
processor_v2 = AutoProcessor.from_pretrained(MODEL)

In [39]:
model_v2 = build_lora_model()

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

trainable params: 7,744,512 || all params: 264,229,440 || trainable%: 2.9310


In [40]:
raw_latex = load_dataset(DATASET, split="train").shuffle(seed=42)
train_latex = raw_latex.select(range(NUM_TRAIN_SAMPLES_2))
eval_ds_v2  = raw_latex.select(range(NUM_TRAIN_SAMPLES_2, NUM_TRAIN_SAMPLES_2 + NUM_EVAL_SAMPLES))
train_latex = train_latex.select_columns(["image", "text"])
eval_ds_v2  = eval_ds_v2.select_columns(["image", "text"])
print(f"LaTeX_OCR train samples : {len(train_latex)}")

LaTeX_OCR train samples : 400


In [41]:
raw_math = load_dataset(DATASET_MATHWRITING, split="train").shuffle(seed=42)

math_cols = raw_math.column_names
print("MathWriting columns:", math_cols)

formula_col = (
    "text" if "text" in math_cols else
    "formula" if "formula" in math_cols else math_cols[1]
)
image_col = "image" if "image" in math_cols else math_cols[0]

train_math = raw_math.select(range(MAX_MATHWRITING))
if formula_col != "text" or image_col != "image":
    train_math = train_math.rename_columns({image_col: "image", formula_col: "text"})
train_math = train_math.select_columns(["image", "text"])
print(f"MathWriting train samples: {len(train_math)}")

MathWriting columns: ['image', 'latex', 'sample_id', 'split_tag', 'data_type']
MathWriting train samples: 400


In [42]:
train_ds_v2 = concatenate_datasets([train_latex, train_math]).shuffle(seed=42)
print(f"Combined train: {len(train_ds_v2)} samples  |  Eval: {len(eval_ds_v2)} samples")

Combined train: 800 samples  |  Eval: 70 samples


In [43]:
collate_fn_v2 = make_collate_fn(processor_v2)
test_batch_v2 = collate_fn_v2([train_ds_v2[0], train_ds_v2[1]])
print({k: v.shape for k, v in test_batch_v2.items()})

{'pixel_values': torch.Size([2, 5, 3, 512, 512]), 'pixel_attention_mask': torch.Size([2, 5, 512, 512]), 'input_ids': torch.Size([2, 448]), 'attention_mask': torch.Size([2, 448]), 'labels': torch.Size([2, 448])}


In [44]:
training_args_v2 = build_training_args(OUTPUT_DIR_V2)

trainer_v2 = SFTTrainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=train_ds_v2,
    eval_dataset=eval_ds_v2,
    data_collator=collate_fn_v2,
    peft_config=None,
)
trainer_v2.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 49279, 'bos_token_id': 1, 'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,0.352020,0.424141


TrainOutput(global_step=100, training_loss=0.7543478345870972, metrics={'train_runtime': 3709.7492, 'train_samples_per_second': 0.216, 'train_steps_per_second': 0.027, 'total_flos': 839463544468224.0, 'train_loss': 0.7543478345870972})

In [45]:
test_ds_v2 = raw_latex.select(range(
    NUM_TRAIN_SAMPLES_2 + NUM_EVAL_SAMPLES,
    NUM_TRAIN_SAMPLES_2 + NUM_EVAL_SAMPLES + NUM_EVAL_SAMPLES,
))
print(f"Test set size: {len(test_ds_v2)}")

Test set size: 70


In [46]:
results_v2 = evaluate_model(
    model_v2, processor_v2,
    test_ds_v2,
    mode="zero_shot",
)


Model evaluation: zero_shot


{'zero_shot'}: 100%|██████████| 70/70 [09:45<00:00,  8.37s/it]

Result of evaluating zero_shot: CER=0.6454


In [50]:
generate_report_sft_v2(results_v2)